In [10]:
import pandas as pd
import os

data_dir = r'C:\Users\rmsgu\Desktop\project4\data'
park_file = os.path.join(data_dir, '공원.csv')
output_file = os.path.join(data_dir, '공원_시군구별_개수.csv')

df_park = pd.read_csv(park_file, encoding='utf-8')

def extract_sigungu(row):
    address = row.get('소재지지번주소')
    if pd.isna(address) or not isinstance(address, str):
        address = row.get('소재지도로명주소')
    if pd.isna(address) or not isinstance(address, str):
        return 'Unknown'

    # 기본 시도 + 시군구 추출
    parts = address.split()
    if len(parts) < 2:
        return address

    sido = parts[0]
    sigungu = parts[1]

    full_sigungu = f"{sido} {sigungu}"

    # 1. '강진군' 단어가 포함되면 '전라남도 강진군'으로 변경
    if '강진군' in address:
        return '전라남도 강진군'

    # 2. '경개도 포천시' → '경기도 포천시'로 변경
    if '경개도 포천시' in address:
        return '경기도 포천시'

    # 3. '수원시 권선구' → '경기도 수원시'로 변경
    if '수원시 권선구' in address:
        return '경기도 수원시'

    if '세종특별자치시' in address:
        return '세종특별자치시'

    if '서울특별시 은평구' in address:
        return '서울특별시 은평구'    

    if '서울특별시 강서구' in address:
        return '서울특별시 강서구' 
        
    if '부산광역시 해운대구' in address:
        return '부산광역시 해운대구'   
        
    if '강원특별자치도양구군' in address:
        return '강원특별자치도 양구군' 
    
    return full_sigungu

df_park['시군구'] = df_park.apply(extract_sigungu, axis=1)

df_park_count = df_park.groupby('시군구').size().reset_index(name='공원수')

df_park_count.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"시군구별 공원 개수 데이터가 저장되었습니다: {output_file}")


시군구별 공원 개수 데이터가 저장되었습니다: C:\Users\rmsgu\Desktop\project4\data\공원_시군구별_개수.csv


In [11]:
import pandas as pd
import os
import numpy as np

data_dir = r'C:\Users\rmsgu\Desktop\project4\data'
park_file = os.path.join(data_dir, '공원.csv')
output_file = os.path.join(data_dir, '공원_시군구별_개수_통합.csv')

df_park = pd.read_csv(park_file, encoding='utf-8')

# 시도명 통일 매핑 예시
sido_name_corrections = {
    '전북특별자치도': '전라북도',
    '제주특별자치도': '제주특별자치도',  # 필요시 통일
    # 필요한 시도명 추가 가능
}

# 읍/면 단위를 시군구 단위로 변환하는 매핑 딕셔너리 (예시 일부)
# 실제로는 전국 읍면에 대해 모두 작성해야 함
eupmyeon_to_sigungu = {
    '충청남도 결성면': '충청남도 공주시',
    '충청남도 대산읍': '충청남도 서산시',
    '충청남도 부석면': '충청남도 부여군',
    # 추가 매핑 필요
}

def normalize_sido(sido):
    return sido_name_corrections.get(sido, sido)

def extract_address(row):
    addr = row.get('소재지지번주소')
    if pd.isna(addr) or not isinstance(addr, str):
        addr = row.get('소재지도로명주소')
    return addr

def extract_sigungu(row):
    addr = extract_address(row)
    if not addr or not isinstance(addr, str):
        return 'Unknown'

    # 시도명 통일
    parts = addr.split()
    if len(parts) < 2:
        return 'Unknown'

    sido = normalize_sido(parts[0])
    sigungu = parts[1]

    # 전체 주소 정리
    full_addr = f"{sido} {sigungu}"

    # 강진군 보정
    if '강진군' in addr:
        return '전라남도 강진군'

    # 경개도 포천시 보정
    if '경개도 포천시' in addr:
        return '경기도 포천시'

    # 수원시 권선구 → 경기도 수원시
    if '수원시 권선구' in addr:
        return '경기도 수원시'

    # 읍/면 단위 매핑 우선 적용
    if addr in eupmyeon_to_sigungu:
        return eupmyeon_to_sigungu[addr]

    # 읍/면 단위로 되어 있을 수 있으므로 주소 전체로도 매핑 확인
    if addr[:len(addr)] in eupmyeon_to_sigungu:
        return eupmyeon_to_sigungu[addr[:len(addr)]]

    return full_addr

# 시군구 컬럼 생성
df_park['시군구'] = df_park.apply(extract_sigungu, axis=1)

# Unknown 개수 확인
unknown_count = df_park[df_park['시군구'] == 'Unknown'].shape[0]
print(f"'Unknown' 시군구 데이터 개수: {unknown_count}")

# 시군구별 공원 개수 집계
df_park_count = df_park.groupby('시군구').size().reset_index(name='공원수')

# 결과 저장
df_park_count.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"시군구별 공원 개수 데이터가 저장되었습니다: {output_file}")


'Unknown' 시군구 데이터 개수: 0
시군구별 공원 개수 데이터가 저장되었습니다: C:\Users\rmsgu\Desktop\project4\data\공원_시군구별_개수_통합.csv


In [12]:
import pandas as pd
import os

data_dir = r'C:\Users\rmsgu\Desktop\project4\data'
input_file = os.path.join(data_dir, '공원_시군구별_개수.csv')
output_file = os.path.join(data_dir, '공원_시군구별_개수_보정.csv')

df = pd.read_csv(input_file, encoding='utf-8-sig')

# 매핑 딕셔너리: 변경 전 읍/면 → 변경 후 시군구
correction_map = {
    '충청남도 결성면': '충청남도 홍성군',
    '충청남도 부석면': '충청남도 서산시',
    '충청남도 대산읍': '충청남도 서산시',
    '경상남도 거제면': '경상남도 거제시',
}

# 매핑 적용 함수
def apply_correction(sigungu):
    return correction_map.get(sigungu, sigungu)

# 수정된 시군구명 적용
df['시군구'] = df['시군구'].apply(apply_correction)

# 동일 시군구명끼리 합계 재집계
df_corrected = df.groupby('시군구', as_index=False)['공원수'].sum()

# 저장
df_corrected.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"보정된 시군구별 공원 수가 저장되었습니다: {output_file}")


보정된 시군구별 공원 수가 저장되었습니다: C:\Users\rmsgu\Desktop\project4\data\공원_시군구별_개수_보정.csv


In [14]:
import pandas as pd
import os

# 입력 파일 및 저장 경로 설정
data_dir = r'C:\Users\rmsgu\Desktop\project4\data'
output_dir = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data'
input_file = os.path.join(data_dir, 'A정신질환_진료_통계.csv')
output_file = os.path.join(output_dir, '정신질환_지역별.csv')

# 출력 폴더가 없으면 생성
os.makedirs(output_dir, exist_ok=True)

# CSV 파일 읽기
df = pd.read_csv(input_file, encoding='cp949')

# 시도, 시군구 컬럼명 확인 (예: '시도', '시군구' 로 가정)
# print(df.columns)

# 시도, 시군구별 환자수와 입내원일수 합계 계산
df_grouped = df.groupby(['시도', '시군구'], as_index=False).agg({
    '환자수': 'sum',
    '입내원일수': 'sum'
})

# 결과 저장
df_grouped.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"시도, 시군구별 환자수 및 입내원일수 합산 결과가 저장되었습니다: {output_file}")


시도, 시군구별 환자수 및 입내원일수 합산 결과가 저장되었습니다: C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv


In [15]:
import pandas as pd
import os

# 파일 경로 설정
data_dir = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data'
input_file = os.path.join(data_dir, '정신질환_지역별.csv')
output_file = os.path.join(data_dir, '정신질환_지역별_수정.csv')

# CSV 읽기
df = pd.read_csv(input_file, encoding='utf-8-sig')

def process_sigungu(row):
    sido = row['시도']
    sigungu = row['시군구']

    # 1. 시군구가 시도 단어로 시작하면 시도 단어 제거
    if sigungu.startswith(sido):
        sigungu = sigungu[len(sido):].strip()

    # 2. 시도 단어로 시작하지 않고, '구'로 끝나면
    if not sigungu.startswith(sido) and sigungu.endswith('구'):
        sigungu = sigungu[:2] + '시'

    return sigungu

# 시군구 처리
df['시군구'] = df.apply(process_sigungu, axis=1)

# 시도, 시군구별로 환자수, 입내원일수 합산
df_grouped = df.groupby(['시도', '시군구'], as_index=False).agg({
    '환자수': 'sum',
    '입내원일수': 'sum'
})

# 결과 저장
df_grouped.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"수정된 정신질환 지역별 데이터가 저장되었습니다: {output_file}")


수정된 정신질환 지역별 데이터가 저장되었습니다: C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별_수정.csv


In [17]:
import pandas as pd
import os
import re

file_path = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# CSV 읽기 (인코딩 cp949)
df = pd.read_csv(file_path, encoding='cp949')

# 1. 행정구역 열에서 괄호 및 괄호 안 숫자 제거
def remove_parentheses(text):
    return re.sub(r'\s*\(\d+\)', '', str(text))

df['행정구역'] = df['행정구역'].apply(remove_parentheses)

# 2. '2024년08월_계_총인구수' 컬럼 쉼표 제거 후 정수 변환
df['2024년08월_계_총인구수'] = df['2024년08월_계_총인구수'].str.replace(',', '').astype(int)

# 3. 인구수가 0인 행 제거
df = df[df['2024년08월_계_총인구수'] != 0]

# 4. 원본 파일 덮어쓰기
df.to_csv(file_path, index=False, encoding='cp949')

print("파일 수정 완료:", file_path)


파일 수정 완료: C:\Users\rmsgu\Desktop\project4\인구1.csv


In [19]:
import pandas as pd
import os
import re

file_path = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

df = pd.read_csv(file_path, encoding='cp949')

def remove_parentheses(text):
    return re.sub(r'\s*\(\d+\)', '', str(text))

df['행정구역'] = df['행정구역'].apply(remove_parentheses)

df['word_count'] = df['행정구역'].str.split().apply(len)
df = df[df['word_count'] == 2].copy()
df[['시도', '시군구']] = df['행정구역'].str.split(n=1, expand=True)

# 수정 부분
df['인구수'] = df['2024년08월_계_총인구수'].astype(str).str.replace(',', '').astype(int)

df.drop(columns=['행정구역', '2024년08월_계_총인구수', 'word_count'], inplace=True)

df.to_csv(file_path, index=False, encoding='cp949')

print("파일 수정 및 저장 완료:", file_path)


파일 수정 및 저장 완료: C:\Users\rmsgu\Desktop\project4\인구1.csv


In [23]:
import pandas as pd
import os

file_path = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 시도명 정식명칭 → 2글자 약칭 매핑 딕셔너리
sido_mapping = {
    '서울특별시': '서울',
    '부산광역시': '부산',
    '대구광역시': '대구',
    '인천광역시': '인천',
    '광주광역시': '광주',
    '대전광역시': '대전',
    '울산광역시': '울산',
    '세종특별자치시': '세종',
    '경기도': '경기',
    '강원특별자치도': '강원',
    '충청북도': '충북',
    '충청남도': '충남',
    '전북특별자치도': '전북',
    '전라남도': '전남',
    '경상북도': '경북',
    '경상남도': '경남',
    '제주특별자치도': '제주',
}

# CSV 파일 읽기
df = pd.read_csv(file_path, encoding='cp949')

# 시도명 2글자 약칭으로 변환
df['시도'] = df['시도'].map(sido_mapping).fillna(df['시도'])

# 파일 덮어쓰기
df.to_csv(file_path, index=False, encoding='cp949')

print("시도명 2글자 변환 및 저장 완료:", file_path)


시도명 2글자 변환 및 저장 완료: C:\Users\rmsgu\Desktop\project4\인구1.csv


In [35]:
import pandas as pd
import numpy as np

mental_file = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'
output_file = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별_인구대비.csv'

# 정신질환 데이터 불러오기 (인코딩 예외 처리)
try:
    df_mental = pd.read_csv(mental_file, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_mental = pd.read_csv(mental_file, encoding='cp949')

# 인구 데이터 불러오기 (cp949 인코딩 가정)
df_pop = pd.read_csv(population_file, encoding='cp949')

# 문자열 공백 제거
df_mental['시도'] = df_mental['시도'].str.strip()
df_mental['시군구'] = df_mental['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 병합 (정신질환 기준 left join)
df = pd.merge(df_mental, df_pop, on=['시도', '시군구'], how='left')

# 인구수 결측은 -1로 표시
df['인구수'] = df['인구수'].fillna(-1)

# 인구수가 0보다 클 때만 인구 10만명 당 환자수 계산
df['인구_10만명_환자수'] = np.where(
    df['인구수'] > 0,
    df['환자수'] / df['인구수'] * 100000,
    np.nan
)

# 입내원일수 대비 환자수 비율 (평균 입내원일수)
df['평균_입내원일수'] = np.where(
    df['환자수'] > 0,
    df['입내원일수'] / df['환자수'],
    np.nan
)

# unknown 인구수 처리 (별도 행 추가 or 시도, 시군구에 'unknown' 할당)
unknown_rows = df[df['인구수'] == -1].copy()
if not unknown_rows.empty:
    unknown_rows['시도'] = 'unknown'
    unknown_rows['시군구'] = 'unknown'
    unknown_rows['인구수'] = np.nan
    unknown_rows['인구_10만명_환자수'] = np.nan
    unknown_rows['평균_입내원일수'] = np.nan

    df = df[df['인구수'] != -1]
    df = pd.concat([df, unknown_rows], ignore_index=True)

# 결과 저장 (cp949로 저장)
df.to_csv(output_file, index=False, encoding='cp949')

print("인구 대비 정신질환 데이터 저장 완료:", output_file)


인구 대비 정신질환 데이터 저장 완료: C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별_인구대비.csv


In [36]:
import pandas as pd
import numpy as np

mental_file = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 인코딩 처리 (utf-8-sig 우선 시도, 실패시 cp949)
try:
    df_mental = pd.read_csv(mental_file, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_mental = pd.read_csv(mental_file, encoding='cp949')

df_pop = pd.read_csv(population_file, encoding='cp949')

df_mental['시도'] = df_mental['시도'].str.strip()
df_mental['시군구'] = df_mental['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 병합
df = pd.merge(df_mental, df_pop, on=['시도', '시군구'], how='left')

# unknown 처리 전 원본 unknown 행 저장
unknown_original = df[df['인구수'].isna()][['시도', '시군구']].copy()

# 인구수 결측은 -1로 표시
df['인구수'] = df['인구수'].fillna(-1)

# 기존 코드와 동일하게 처리...

# unknown 인구수 처리 시, 원본 시도/시군구 출력
print("인구수 정보 없는 지역 (unknown 처리 예정):")
print(unknown_original)

# 이후 필요시 unknown 처리 추가 가능


인구수 정보 없는 지역 (unknown 처리 예정):
Empty DataFrame
Columns: [시도, 시군구]
Index: []


In [28]:
import pandas as pd

# 파일 경로 설정
mental_file = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 데이터 읽기
try:
    df_mental = pd.read_csv(mental_file, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_mental = pd.read_csv(mental_file, encoding='cp949')

df_pop = pd.read_csv(population_file, encoding='cp949')

# 문자열 공백 제거
df_mental['시도'] = df_mental['시도'].str.strip()
df_mental['시군구'] = df_mental['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 인구 데이터에서 시군구 앞 두 글자 추출 (비교용)
df_pop['시군구_앞두글자'] = df_pop['시군구'].str[:2]

# 정신질환 데이터에도 시군구 앞 두 글자 추가
df_mental['시군구_앞두글자'] = df_mental['시군구'].str[:2]

# 대체할 새 시군구명을 저장할 컬럼 생성(초기값은 기존 시군구)
df_mental['시군구_대체'] = df_mental['시군구']

# 대응되는 인구 데이터의 시군구명을 찾아 대체
def replace_sigungu(row):
    sido = row['시도']
    prefix = row['시군구_앞두글자']
    candidates = df_pop[(df_pop['시도'] == sido) & (df_pop['시군구_앞두글자'] == prefix)]
    if not candidates.empty:
        # 첫 번째 일치하는 시군구명으로 대체
        return candidates.iloc[0]['시군구']
    else:
        return None  # 대응되는 게 없음

df_mental['대체명'] = df_mental.apply(replace_sigungu, axis=1)

# 대체명으로 시군구 대체 (대체명 있으면 그걸로, 없으면 원래값 유지)
df_mental['시군구_대체'] = df_mental['대체명'].combine_first(df_mental['시군구'])

# 대응 안된 행 출력
no_match = df_mental[df_mental['대체명'].isnull()][['시도', '시군구']]
if not no_match.empty:
    print("대응되는 인구 데이터 시군구가 없는 행들:")
    print(no_match)

# 시군구를 대체명으로 변경 후 불필요 컬럼 제거
df_mental['시군구'] = df_mental['시군구_대체']
df_mental.drop(columns=['시군구_앞두글자', '시군구_대체', '대체명'], inplace=True)

# 변경 내용 저장 (덮어쓰기)
df_mental.to_csv(mental_file, index=False, encoding='cp949')

print(f"시군구명 대체 완료 및 저장: {mental_file}")


대응되는 인구 데이터 시군구가 없는 행들:
     시도  시군구
71   경북  군위군
123  부산  진구시
150  세종    시
시군구명 대체 완료 및 저장: C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv


In [30]:
import pandas as pd

# 파일 경로
mental_file = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 파일 읽기 (인코딩 상황에 맞게 변경)
try:
    df_mental = pd.read_csv(mental_file, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_mental = pd.read_csv(mental_file, encoding='cp949')

df_pop = pd.read_csv(population_file, encoding='cp949')

# 문자열 공백 제거
df_mental['시도'] = df_mental['시도'].str.strip()
df_mental['시군구'] = df_mental['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 정신질환 기준 지역셋
mental_regions = set(zip(df_mental['시도'], df_mental['시군구']))

# 인구 데이터 기준 지역셋
pop_regions = set(zip(df_pop['시도'], df_pop['시군구']))

# 정신질환 데이터에만 있는 지역 (인구 데이터와 불일치)
mental_only = mental_regions - pop_regions

# 인구 데이터에만 있는 지역 (정신질환 데이터와 불일치)
pop_only = pop_regions - mental_regions

print("=== 정신질환 데이터에만 있는 지역 (대응 안 됨) ===")
for region in sorted(mental_only):
    print(region)

print("\n=== 인구 데이터에만 있는 지역 (대응 안 됨) ===")
for region in sorted(pop_only):
    print(region)


=== 정신질환 데이터에만 있는 지역 (대응 안 됨) ===

=== 인구 데이터에만 있는 지역 (대응 안 됨) ===


In [34]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv'

# 인코딩 예외 처리하며 읽기
try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

# 시도, 시군구 컬럼 기준으로 중복 제거 (첫 번째 행은 남김)
df = df.drop_duplicates(subset=['시도', '시군구'], keep='first')

# 파일 덮어쓰기
df.to_csv(file_path, index=False, encoding='cp949')

print(f"중복 제거 후 파일 저장 완료: {file_path}")


중복 제거 후 파일 저장 완료: C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv


In [38]:
import pandas as pd
import os

input_file = r'C:\Users\rmsgu\Desktop\project4\data\공원_시군구별_개수.csv'
output_dir = r'C:\Users\rmsgu\Desktop\project4\processed_data'
output_file = os.path.join(output_dir, '공원_시군구별_개수_가공.csv')

# 시도명 2글자 약칭 매핑
sido_mapping = {
    '서울특별시': '서울',
    '부산광역시': '부산',
    '대구광역시': '대구',
    '인천광역시': '인천',
    '광주광역시': '광주',
    '대전광역시': '대전',
    '울산광역시': '울산',
    '세종특별자치시': '세종',
    '경기도': '경기',
    '강원특별자치도': '강원',
    '충청북도': '충북',
    '충청남도': '충남',
    '전북특별자치도': '전북',
    '전라남도': '전남',
    '경상북도': '경북',
    '경상남도': '경남',
    '제주특별자치도': '제주',
}

# 데이터 읽기
df = pd.read_csv(input_file, encoding='utf-8-sig')

# 시군구를 공백 기준 두 단어로 분리 ('시도'와 '시군구'로 분리)
split_cols = df['시군구'].str.split(' ', n=1, expand=True)
df['시도'] = split_cols[0]
df['시군구'] = split_cols[1]

# 시도명을 2글자 약칭으로 변환
df['시도'] = df['시도'].map(sido_mapping).fillna(df['시도'])

# 기존 '시군구' 컬럼 재설정 후, 기존 원래 '시군구' 컬럼(분리 전) 삭제
# (분리 후 덮어쓴 상태라 삭제 불필요)

# 필요한 컬럼 순서 조정 (예: 시도, 시군구, 공원수)
df = df[['시도', '시군구', '공원수']]

# 출력 디렉토리 생성
os.makedirs(output_dir, exist_ok=True)

# 저장
df.to_csv(output_file, index=False, encoding='cp949')

print(f"가공 완료 및 저장: {output_file}")


가공 완료 및 저장: C:\Users\rmsgu\Desktop\project4\processed_data\공원_시군구별_개수_가공.csv


In [43]:
import pandas as pd

input_file = r'C:\Users\rmsgu\Desktop\project4\data\평균소득월액.csv'
output_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\평균소득월액_가공.csv'

sido_mapping = {
    '서울특별시': '서울',
    '부산광역시': '부산',
    '대구광역시': '대구',
    '인천광역시': '인천',
    '광주광역시': '광주',
    '대전광역시': '대전',
    '울산광역시': '울산',
    '세종특별자치시': '세종',
    '경기도': '경기',
    '강원특별자치도': '강원',
    '충청북도': '충북',
    '충청남도': '충남',
    '전라북도': '전북',
    '전라남도': '전남',
    '경상북도': '경북',
    '경상남도': '경남',
    '제주특별자치도': '제주',
    '전북특별자치도': '전북'
}

df = pd.read_csv(input_file, encoding='cp949')

def split_sido_sigungu(address):
    for full_sido in sido_mapping.keys():
        if full_sido in address:
            sido_short = sido_mapping[full_sido]
            idx = address.find(full_sido)
            sigungu = address[idx + len(full_sido):].strip()
            return pd.Series([sido_short, sigungu])
    return pd.Series(['unknown', address])

df[['시도', '시군구']] = df['시군구'].apply(split_sido_sigungu)

# 기준년월 열 삭제
df.drop(columns=['기준년월'], inplace=True)

# 필요한 컬럼만 선택하여 저장
cols = ['시도', '시군구', '평균소득월액']
df = df[cols]

df.to_csv(output_file, index=False, encoding='cp949')


In [44]:
import pandas as pd
import os

input_file = r'C:\Users\rmsgu\Desktop\project4\data\진학률.csv'
output_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\진학률_가공.csv'

# 데이터 읽기 (인코딩 확인 필요)
df = pd.read_csv(input_file, encoding='cp949')

# 필요한 열만 선택
df_selected = df[['시도', '시군구', '진학률']]

# 저장할 폴더가 없으면 생성
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 저장
df_selected.to_csv(output_file, index=False, encoding='cp949')


In [45]:
import pandas as pd
import os

input_file = r'C:\Users\rmsgu\Desktop\project4\data\1인세대수.csv'
output_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\1인세대수_시군구별.csv'

# 시도명 줄임말 매핑
sido_mapping = {
    '서울특별시': '서울',
    '부산광역시': '부산',
    '대구광역시': '대구',
    '인천광역시': '인천',
    '광주광역시': '광주',
    '대전광역시': '대전',
    '울산광역시': '울산',
    '세종특별자치시': '세종',
    '경기도': '경기',
    '강원특별자치도': '강원',
    '충청북도': '충북',
    '충청남도': '충남',
    '전라북도': '전북',
    '전라남도': '전남',
    '경상북도': '경북',
    '경상남도': '경남',
    '제주특별자치도': '제주',
    '전북특별자치도': '전북'  # 포함
}

# 1. CSV 파일 읽기 (인코딩 확인 필요)
df = pd.read_csv(input_file, encoding='cp949')

# 2. 불필요한 열 삭제
df.drop(columns=['행정기관코드', '기준연월', '읍면동명'], inplace=True)

# 3. 시도명 줄임말 적용
df['시도'] = df['시도명'].map(sido_mapping)

# 4. 시군구명 복사
df['시군구'] = df['시군구명']

# 5. 시도명, 시군구명 기준 그룹화 후 '계' 합산
df_grouped = df.groupby(['시도', '시군구'], as_index=False)['계'].sum()

# 6. 열 이름 정리
df_grouped.rename(columns={'계': '1인세대수'}, inplace=True)

# 7. 저장 폴더 생성
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# 8. 결과 저장
df_grouped.to_csv(output_file, index=False, encoding='cp949')

print(f"파일 저장 완료: {output_file}")


파일 저장 완료: C:\Users\rmsgu\Desktop\project4\processed_data\1인세대수_시군구별.csv


In [46]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\1인세대수_시군구별.csv'

# 데이터 불러오기
df = pd.read_csv(file_path, encoding='cp949')

# 시군구가 2단어 이상인 경우 첫 단어로 변경
def first_word_only(s):
    parts = str(s).split()
    if len(parts) >= 2:
        return parts[0]
    else:
        return s

df['시군구'] = df['시군구'].apply(first_word_only)

# 시도, 시군구별로 다시 그룹화하여 1인세대수 합산
df_grouped = df.groupby(['시도', '시군구'], as_index=False)['1인세대수'].sum()

# 결과 덮어쓰기 저장
df_grouped.to_csv(file_path, index=False, encoding='cp949')

print("파일 업데이트 완료:", file_path)


파일 업데이트 완료: C:\Users\rmsgu\Desktop\project4\processed_data\1인세대수_시군구별.csv


In [53]:
import pandas as pd

input_file = r'C:\Users\rmsgu\Desktop\project4\data\범죄율.csv'
output_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\범죄율_transposed.csv'

# 파일 읽기 (인코딩은 필요에 따라 조정)
df = pd.read_csv(input_file, encoding='cp949')

# 전치
df_t = df.T

# 전치된 데이터 저장 (index 포함 여부는 필요에 따라 조정)
df_t.to_csv(output_file, index=True, encoding='cp949')

print(f"전치된 파일 저장 완료: {output_file}")


전치된 파일 저장 완료: C:\Users\rmsgu\Desktop\project4\processed_data\범죄율_transposed.csv


In [59]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\범죄율_transposed.csv'

# 1. CSV 파일을 헤더 없이 읽기
df = pd.read_csv(file_path, header=None, encoding='cp949')

# 2. 1행(0번 인덱스)과 3행(2번 인덱스) 제거
df = df.drop([0, 2]).reset_index(drop=True)

# 3. 2행(원래 1번 인덱스)은 범죄대분류(열 이름)
crime_major = df.iloc[0].tolist()

# 4. 데이터 부분은 3행부터 (현재 1행부터)
data = df.iloc[1:].reset_index(drop=True)

# 5. 열 이름으로 범죄대분류 설정
data.columns = crime_major

# 6. 행정구역은 첫 번째 열(원래 0번 열)인데, 지금 데이터 첫 행에 해당하므로 분리해서 변수로 저장
regions = data.iloc[:, 0].tolist()

# 7. 행정구역 열은 데이터프레임에서 따로 분리
data = data.drop(data.columns[0], axis=1)

# 8. 숫자 데이터로 변환 (문자형 숫자 가능성 대비)
data = data.apply(pd.to_numeric, errors='coerce').fillna(0)

# 9. 범죄대분류별로 같은 이름 열끼리 합산
data_grouped = data.groupby(data.columns, axis=1).sum()

# 10. 행정구역 열을 다시 붙임
data_grouped.insert(0, '행정구역', regions)

# 11. 결과를 파일에 덮어쓰기
data_grouped.to_csv(file_path, index=False, encoding='cp949')

print("범죄대분류별 합산 및 행정구역 첫열 추가 완료")


범죄대분류별 합산 및 행정구역 첫열 추가 완료


C:\Users\rmsgu\AppData\Local\Temp\ipykernel_27456\1167513854.py:30: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  data_grouped = data.groupby(data.columns, axis=1).sum()


In [60]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\범죄율_transposed.csv'

# 데이터 불러오기
df = pd.read_csv(file_path, encoding='cp949')

def split_region(region):
    if pd.isna(region):
        return pd.Series([None, None])
    # 예외: 경기도, 강원도
    if region.startswith("경기도") or region.startswith("강원도"):
        sido = region[:2]
        sigungu = region[3:]  # 3번째 글자 버림
    else:
        sido = region[:2]
        sigungu = region[2:]
    return pd.Series([sido, sigungu])

# '행정구역' 열 기준 분리
df[['시도', '시군구']] = df['행정구역'].apply(split_region)

# 원본 '행정구역' 열 삭제
df = df.drop(columns=['행정구역'])

# 컬럼 순서: 시도, 시군구, 나머지 열 순서 유지
cols = ['시도', '시군구'] + [col for col in df.columns if col not in ['시도', '시군구']]
df = df[cols]

# 덮어쓰기
df.to_csv(file_path, index=False, encoding='cp949')

print("시도, 시군구 분리 및 파일 덮어쓰기 완료")


시도, 시군구 분리 및 파일 덮어쓰기 완료


In [64]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\범죄율_transposed.csv'

# 데이터 불러오기
df = pd.read_csv(file_path, encoding='cp949')

# 경북 군위군, 대구 군위군 행 선택
mask_gyeongbuk = (df['시도'] == '경북') & (df['시군구'] == '군위군')
mask_daegu = (df['시도'] == '대구') & (df['시군구'] == '군위군')

if mask_gyeongbuk.any() and mask_daegu.any():
    idx_gb = df[mask_gyeongbuk].index[0]
    idx_dg = df[mask_daegu].index[0]
    
    # 합산할 숫자 열들 (시도, 시군구 제외)
    numeric_cols = df.columns.difference(['시도', '시군구'])
    
    # 두 행 합산 후 대구 군위군에 저장
    df.loc[idx_dg, numeric_cols] = df.loc[[idx_gb, idx_dg], numeric_cols].sum()
    
    # 경북 군위군 행 삭제
    df = df.drop(idx_gb).reset_index(drop=True)
    
    # 파일 덮어쓰기
    df.to_csv(file_path, index=False, encoding='cp949')
    print("경북 군위군과 대구 군위군 합산 후 저장 완료")
else:
    print("경북 군위군 또는 대구 군위군 데이터가 없습니다.")


경북 군위군과 대구 군위군 합산 후 저장 완료


In [68]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\범죄율_transposed.csv'

# 데이터 불러오기
df = pd.read_csv(file_path, encoding='cp949')

# 합칠 범죄열 리스트
cols_to_add = ['병역범죄', '보건범죄', '선거범죄', '안보범죄', '환경범죄']
target_col = '기타범죄'

# '기타범죄' 열에 합산
df[target_col] = df[target_col].fillna(0) + df[cols_to_add].fillna(0).sum(axis=1)

# 합산한 열 삭제
df = df.drop(columns=cols_to_add)

# 각 행별 합계 (숫자 열만) 구해서 '총합' 열 생성
numeric_cols = df.select_dtypes(include='number').columns
df['총합'] = df[numeric_cols].sum(axis=1)

# 파일 덮어쓰기
df.to_csv(file_path, index=False, encoding='cp949')

print("병역, 보건, 선거, 안보, 환경범죄를 기타범죄에 합산하고 총합 열 추가 완료")


병역, 보건, 선거, 안보, 환경범죄를 기타범죄에 합산하고 총합 열 추가 완료


In [71]:
import pandas as pd

# 파일 경로
file_1인세대 = r'C:\Users\rmsgu\Desktop\project4\processed_data\시군구\1인세대수_가공.csv'
file_인구 = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 1인세대수 데이터 불러오기
df_1인 = pd.read_csv(file_1인세대, encoding='cp949')

# 인구 데이터 불러오기
df_pop = pd.read_csv(file_인구, encoding='cp949')

# 인구 컬럼명 통일 (인구수)
df_pop.rename(columns={'2024년08월_계_총인구수': '인구수'}, inplace=True)

# 시도, 시군구 컬럼명 통일
df_pop.rename(columns={'시도명': '시도', '시군구명': '시군구'}, inplace=True)

# 데이터 형식 맞추기 (strip, 공백제거)
df_1인['시도'] = df_1인['시도'].str.strip()
df_1인['시군구'] = df_1인['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 인구와 1인세대수 병합 (left join으로 1인세대수 기준)
df_merged = pd.merge(df_1인, df_pop[['시도', '시군구', '인구수']], on=['시도', '시군구'], how='left')

# 인구수가 없는 곳은 0 또는 NaN 처리 (NaN->0)
df_merged['인구수'] = df_merged['인구수'].fillna(0)

# 인구 10만명당 1인세대수 계산 (인구 0인 경우는 0 처리)
df_merged['인구10만명당_1인세대수'] = df_merged.apply(
    lambda row: (row['1인세대수'] / row['인구수'] * 100000) if row['인구수'] > 0 else 0,
    axis=1
)

# 기존 파일 덮어쓰기
df_merged.to_csv(file_1인세대, index=False, encoding='cp949')

print("인구 10만명당 1인세대수 열 추가 및 파일 덮어쓰기 완료")


인구 10만명당 1인세대수 열 추가 및 파일 덮어쓰기 완료


In [73]:
import pandas as pd

file_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\시군구\1인세대수_가공.csv'

df = pd.read_csv(file_path, encoding='cp949')

# 인구수 열 삭제 (존재하면 삭제)
if '인구수' in df.columns:
    df.drop(columns=['인구수'], inplace=True)

# 덮어쓰기 저장
df.to_csv(file_path, index=False, encoding='cp949')

print("인구수 열 삭제 후 파일 덮어쓰기 완료")


인구수 열 삭제 후 파일 덮어쓰기 완료


In [80]:
import pandas as pd

park_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\시군구\공원_시군구별_개수_가공.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

df_park = pd.read_csv(park_file, encoding='cp949')
df_pop = pd.read_csv(population_file, encoding='cp949')

df_park['시도'] = df_park['시도'].str.strip()
df_park['시군구'] = df_park['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

df_merged = pd.merge(df_park, df_pop[['시도', '시군구', '인구수']], how='left', on=['시도', '시군구'])

df_merged['인구수'] = df_merged['인구수'].fillna(0)

df_merged['인구10만명당_공원수'] = df_merged.apply(
    lambda row: (row['공원수'] / row['인구수'] * 100000) if row['인구수'] > 0 else 0,
    axis=1
)

df_merged.drop(columns=['인구수'], inplace=True)

df_merged.to_csv(park_file, index=False, encoding='cp949')

print("인구10만명당 공원수 열 추가 후 저장 완료")


인구10만명당 공원수 열 추가 후 저장 완료


In [78]:
import pandas as pd

# 파일 경로
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'
park_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\시군구\공원_시군구별_개수_가공.csv'

# 데이터 불러오기 (필요하면 encoding 변경)
df_pop = pd.read_csv(population_file, encoding='cp949')
df_park = pd.read_csv(park_file, encoding='cp949')

# 시도, 시군구 컬럼 공백 제거
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()
df_park['시도'] = df_park['시도'].str.strip()
df_park['시군구'] = df_park['시군구'].str.strip()

# 두 파일의 (시도, 시군구) 집합 생성
pop_keys = set(zip(df_pop['시도'], df_pop['시군구']))
park_keys = set(zip(df_park['시도'], df_park['시군구']))

# 각각 대응되지 않는 시군구 찾기
pop_not_in_park = pop_keys - park_keys
park_not_in_pop = park_keys - pop_keys

# 결과 출력
if pop_not_in_park:
    print("인구1.csv에 있지만 공원_시군구별_개수_가공.csv에 없는 시군구:")
    for sido, sigungu in sorted(pop_not_in_park):
        print(f"{sido} {sigungu}")
else:
    print("인구1.csv의 모든 시군구가 공원_시군구별_개수_가공.csv에 존재합니다.")

print("\n")

if park_not_in_pop:
    print("공원_시군구별_개수_가공.csv에 있지만 인구1.csv에 없는 시군구:")
    for sido, sigungu in sorted(park_not_in_pop):
        print(f"{sido} {sigungu}")
else:
    print("공원_시군구별_개수_가공.csv의 모든 시군구가 인구1.csv에 존재합니다.")


인구1.csv에 있지만 공원_시군구별_개수_가공.csv에 없는 시군구:
인천 옹진군
전북 김제시


공원_시군구별_개수_가공.csv에 있지만 인구1.csv에 없는 시군구:
서울 아현동


In [79]:
import pandas as pd

# 파일 경로
park_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\시군구\공원_시군구별_개수_가공.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 데이터 불러오기
df_park = pd.read_csv(park_file, encoding='cp949')
df_pop = pd.read_csv(population_file, encoding='cp949')

# 공백 제거
df_park['시도'] = df_park['시도'].str.strip()
df_park['시군구'] = df_park['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 1) '서울 아현동' 행 삭제
df_park = df_park[~((df_park['시도'] == '서울') & (df_park['시군구'] == '아현동'))]

# 2) '인천 옹진군', '전북 김제시' 공원수 0으로 추가 (존재하지 않는 경우에만)
missing_rows = []
for sido, sigungu in [('인천', '옹진군'), ('전북', '김제시')]:
    if not ((df_park['시도'] == sido) & (df_park['시군구'] == sigungu)).any():
        missing_rows.append({'시도': sido, '시군구': sigungu, '공원수': 0})

if missing_rows:
    df_park = pd.concat([df_park, pd.DataFrame(missing_rows)], ignore_index=True)

# 3) 파일 덮어쓰기
df_park.to_csv(park_file, index=False, encoding='cp949')

print("아현동 삭제 및 누락된 시군구 공원수 0 추가 완료")


아현동 삭제 및 누락된 시군구 공원수 0 추가 완료


In [81]:
import pandas as pd

# 파일 경로
crime_file = r'C:\Users\rmsgu\Desktop\project4\processed_data\시군구\범죄율_가공.csv'
population_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'

# 데이터 불러오기
df_crime = pd.read_csv(crime_file, encoding='cp949')
df_pop = pd.read_csv(population_file, encoding='cp949')

# 시도, 시군구 공백 제거
df_crime['시도'] = df_crime['시도'].str.strip()
df_crime['시군구'] = df_crime['시군구'].str.strip()
df_pop['시도'] = df_pop['시도'].str.strip()
df_pop['시군구'] = df_pop['시군구'].str.strip()

# 인구수 병합
df_merged = pd.merge(df_crime, df_pop[['시도', '시군구', '인구수']], on=['시도', '시군구'], how='left')

# 인구수 없는 곳은 0으로 대체
df_merged['인구수'] = df_merged['인구수'].fillna(0)

# 인구 10만명당 범죄율 계산 대상 열들
crime_columns = ['강력범죄', '교통범죄', '기타범죄', '마약범죄', '절도범죄', '지능범죄', '특별경제범죄', '폭력범죄', '풍속범죄', '총합']

# 각 열에 대해 인구 10만명당 범죄율 새 열 생성
for col in crime_columns:
    new_col = f'{col}_10만명당'
    df_merged[new_col] = df_merged.apply(
        lambda row: (row[col] / row['인구수'] * 100000) if row['인구수'] > 0 else 0,
        axis=1
    )

# 불필요한 '인구수' 열 삭제
df_merged.drop(columns=['인구수'], inplace=True)

# 파일 덮어쓰기
df_merged.to_csv(crime_file, index=False, encoding='cp949')

print("인구 10만명당 범죄율 계산 및 파일 덮어쓰기 완료")


인구 10만명당 범죄율 계산 및 파일 덮어쓰기 완료


In [82]:
import pandas as pd
import os

# 파일 경로
base_dir = r'C:\Users\rmsgu\Desktop\project4'
mental_file = os.path.join(base_dir, 'mental_illness_data', '정신질환_지역별_인구대비.csv')
processed_dir = os.path.join(base_dir, 'processed_data', '시군구')

# 각 가공 파일과 가져올 열 이름 매핑
files_columns = {
    '1인세대수_가공.csv': '인구10만명당_1인세대수',
    '공원_시군구별_개수_가공.csv': '인구10만명당_공원수',
    '범죄율_가공.csv': '총합_10만명당',
    '진학률_가공.csv': '진학률',
    '평균소득월액_가공.csv': '평균소득월액',
}

# 정신질환 데이터 불러오기
df_mental = pd.read_csv(mental_file, encoding='cp949')

# 시도, 시군구 문자열 앞뒤 공백 제거 (혹시 모르니)
df_mental['시도'] = df_mental['시도'].str.strip()
df_mental['시군구'] = df_mental['시군구'].str.strip()

# 각 가공 파일을 하나씩 읽고 병합
for filename, colname in files_columns.items():
    file_path = os.path.join(processed_dir, filename)
    df_tmp = pd.read_csv(file_path, encoding='cp949')
    # 공백 제거
    df_tmp['시도'] = df_tmp['시도'].str.strip()
    df_tmp['시군구'] = df_tmp['시군구'].str.strip()

    # 필요한 열만 남기기
    if colname not in df_tmp.columns:
        raise ValueError(f"파일 {filename}에 '{colname}' 열이 없습니다.")
    df_tmp = df_tmp[['시도', '시군구', colname]]

    # 정신질환 데이터에 병합 (left join 유지)
    df_mental = df_mental.merge(df_tmp, on=['시도', '시군구'], how='left')

# 결과 저장 (덮어쓰기 아님)
output_file = os.path.join(base_dir, 'processed_data', 'data_시군구.csv')
df_mental.to_csv(output_file, index=False, encoding='cp949')

print(f"파일이 '{output_file}'로 저장되었습니다.")


파일이 'C:\Users\rmsgu\Desktop\project4\processed_data\data_시군구.csv'로 저장되었습니다.


In [83]:
import pandas as pd

# 파일 경로 설정
input_file = r'C:\Users\rmsgu\Desktop\project4\인구1.csv'
output_file = r'C:\Users\rmsgu\Desktop\project4\인구2.csv'

# 데이터 불러오기
df = pd.read_csv(input_file, encoding='cp949')

# 시도별 인구수 합산 (인구수 컬럼 이름이 다르면 확인 필요)
df_grouped = df.groupby('시도', as_index=False)['인구수'].sum()

# 결과 저장
df_grouped.to_csv(output_file, index=False, encoding='cp949')

print("시도별 인구수 합산 완료 및 인구2.csv 저장됨")


시도별 인구수 합산 완료 및 인구2.csv 저장됨


In [88]:
import pandas as pd

file_godoksa = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\고독사.csv'
file_population = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\인구2.csv'

# UTF-8 인코딩으로 읽기 시도
try:
    df_godoksa = pd.read_csv(file_godoksa, encoding='utf-8')
except UnicodeDecodeError:
    # UTF-8 실패 시 CP949로 재시도
    df_godoksa = pd.read_csv(file_godoksa, encoding='cp949')

try:
    df_population = pd.read_csv(file_population, encoding='utf-8')
except UnicodeDecodeError:
    df_population = pd.read_csv(file_popㅁulation, encoding='cp949')

# 시도 기준으로 병합
df_merged = pd.merge(df_godoksa, df_population[['시도', '인구수']], on='시도', how='left')

# 인구 10만명당 고독사 계산
df_merged['인구_10만명당_고독사'] = (df_merged['평균'] / df_merged['인구수']) * 100000

# 덮어쓰기 저장 (기존 파일 덮어쓰기)
df_merged.to_csv(file_godoksa, index=False, encoding='utf-8-sig')


In [91]:
import pandas as pd

file_consult = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\상담현황.csv'
file_population = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\인구2.csv'

# 인코딩은 상황에 따라 바꾸세요 (utf-8 또는 cp949)
try:
    df_consult = pd.read_csv(file_consult, encoding='utf-8')
except UnicodeDecodeError:
    df_consult = pd.read_csv(file_consult, encoding='cp949')

try:
    df_population = pd.read_csv(file_population, encoding='utf-8')
except UnicodeDecodeError:
    df_population = pd.read_csv(file_population, encoding='cp949')

# 시도 기준 병합
df_merged = pd.merge(df_consult, df_population[['시도', '인구수']], on='시도', how='left')

# 인구 10만명당 상담건수 계산
df_merged['인구_10만명당_상담건수'] = (df_merged['상담건수'] / df_merged['인구수']) * 100000

# 덮어쓰기 저장 (utf-8-sig로 저장하면 엑셀에서 한글 깨짐 방지)
df_merged.to_csv(file_consult, index=False, encoding='utf-8-sig')


In [92]:
import pandas as pd

file_institution = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\정신건강_관련_기관_설치_현황.csv'
file_population = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\인구2.csv'

# 인코딩 문제 대비
try:
    df_institution = pd.read_csv(file_institution, encoding='utf-8')
except UnicodeDecodeError:
    df_institution = pd.read_csv(file_institution, encoding='cp949')

try:
    df_population = pd.read_csv(file_population, encoding='utf-8')
except UnicodeDecodeError:
    df_population = pd.read_csv(file_population, encoding='cp949')

# 병합
df_merged = pd.merge(df_institution, df_population[['시도', '인구수']], on='시도', how='left')

# 인구 10만명당 기관수 계산
df_merged['인구_10만명당_기관수'] = (df_merged['개수'] / df_merged['인구수']) * 100000

# 파일 덮어쓰기 (utf-8-sig로 저장하면 엑셀에서 한글 깨짐 방지)
df_merged.to_csv(file_institution, index=False, encoding='utf-8-sig')


In [93]:
import pandas as pd

# 파일 경로
file_path = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별.csv'
output_path = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별_시도.csv'

# 파일 불러오기 (인코딩 문제 있을 경우 조정)
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

# 시도별 환자수와 입내원일수 합산
df_grouped = df.groupby('시도', as_index=False)[['환자수', '입내원일수']].sum()

# 결과 저장 (utf-8-sig는 엑셀에서 한글 깨짐 방지)
df_grouped.to_csv(output_path, index=False, encoding='utf-8-sig')


In [94]:
import pandas as pd

# 파일 경로
file_path = r'C:\Users\rmsgu\Desktop\project4\mental_illness_data\정신질환_지역별_시도.csv'
population_path = r'C:\Users\rmsgu\Desktop\project4\processed_data\시도\인구2.csv'

# 파일 불러오기
try:
    df_mental = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df_mental = pd.read_csv(file_path, encoding='cp949')

try:
    df_pop = pd.read_csv(population_path, encoding='utf-8')
except UnicodeDecodeError:
    df_pop = pd.read_csv(population_path, encoding='cp949')

# 시도별 인구수 병합
df_merged = pd.merge(df_mental, df_pop[['시도', '인구수']], on='시도', how='left')

# 인구 10만명당 환자수 계산
df_merged['인구10만명당_환자수'] = df_merged.apply(
    lambda row: (row['환자수'] / row['인구수'] * 100000) if row['인구수'] > 0 else 0,
    axis=1
)

# 평균 입내원일수 계산 (입내원일수 ÷ 환자수, 환자수 0인 경우 0 처리)
df_merged['평균입내원일수'] = df_merged.apply(
    lambda row: (row['입내원일수'] / row['환자수']) if row['환자수'] > 0 else 0,
    axis=1
)

# 기존 파일 덮어쓰기 (utf-8-sig 인코딩으로 한글 깨짐 방지)
df_merged.to_csv(file_path, index=False, encoding='utf-8-sig')


In [95]:
import pandas as pd

# 기본 경로
base_path = r'C:\Users\rmsgu\Desktop\project4\processed_data'
subfolder = base_path + r'\시도'

# 병합 대상 파일과 열 목록
files_columns = {
    '1인당_정신건강_예산.csv': ['시도', '1인당 정신건강예산'],
    '고독사.csv': ['시도', '인구_10만명당_고독사'],
    '상담현황.csv': ['시도', '인구_10만명당_상담건수'],
    '정신건강_관련_기관_설치_현황.csv': ['시도', '인구_10만명당_기관수'],
    '정신건강_관련_기관_인구_10만_명당_인력_수.csv': ['시도', '상근인력대비 전문인력 비율', '인구 10만명당 상근인력', '인구 10만명당 전문인력']
}

# data_시도.csv 경로
data_si_file = base_path + r'\data_시도.csv'

# 1. 원본 데이터 불러오기 (인코딩 문제 대비)
try:
    df_data = pd.read_csv(data_si_file, encoding='utf-8')
except UnicodeDecodeError:
    df_data = pd.read_csv(data_si_file, encoding='cp949')

# 2. 각 파일에서 필요한 열만 읽어와 순차 병합
for filename, cols in files_columns.items():
    filepath = f"{subfolder}\\{filename}"
    try:
        df_temp = pd.read_csv(filepath, encoding='utf-8', usecols=cols)
    except UnicodeDecodeError:
        df_temp = pd.read_csv(filepath, encoding='cp949', usecols=cols)

    # 시도 컬럼 strip 처리 (공백 제거) 혹시 몰라
    df_temp['시도'] = df_temp['시도'].str.strip()
    df_data['시도'] = df_data['시도'].str.strip()

    # 병합 (left join으로 원본 기준 유지)
    df_data = pd.merge(df_data, df_temp, on='시도', how='left')

# 3. 결과 저장 (utf-8-sig로 한글 깨짐 최소화)
df_data.to_csv(data_si_file, index=False, encoding='utf-8-sig')
